# AIGC 5005 · Module 05 · Part 2
## Splitting data and avoiding leakage

Companion notebook for the matching part of the Module 05 learning content. Read each explanation, **predict** what the next cell will show, run it, and compare. Change values and run again; you cannot break anything.

**Kernel:** choose the `module05-sklearn` conda environment (top right in VS Code). To make it: `conda create -n module05-sklearn python=3.11 ipykernel`, then `conda activate module05-sklearn` and `pip install -r requirements.txt`. 

## Stratified split

In [1]:
from customers import make_customers
from sklearn.model_selection import train_test_split

customers = make_customers()
X = customers.drop(columns="churned")
y = customers["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(1500, 5) (500, 5)
0.141 0.142


## Without `stratify`
The test churn rate depends on the luck of the shuffle.

In [2]:
for seed in (1, 2, 3, 4):
    _, _, _, y_te = train_test_split(X, y, test_size=0.25, random_state=seed)
    print("no stratify, seed", seed, y_te.mean().round(3))

no stratify, seed 1 0.136
no stratify, seed 2 0.15
no stratify, seed 3 0.152
no stratify, seed 4 0.132


## Data leakage
The data below is **pure noise**, so an honest score is about 0.5. In the leaky version the columns are chosen using every row *before* cross-validation. In the honest version the selection is inside a pipeline, so it is redone in each training fold.

In [3]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(1)
X_noise = rng.normal(size=(100, 1000))
y_noise = rng.integers(0, 2, 100)

X_leaky = SelectKBest(f_classif, k=10).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(), X_leaky, y_noise, cv=5).mean()

pipe = make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression())
honest = cross_val_score(pipe, X_noise, y_noise, cv=5).mean()
print(round(leaky, 2), round(honest, 2))

0.85 0.47


Nothing raised an error, which is exactly why leakage is dangerous. The leaky score looks great and is meaningless.

## Try it
Change the seed in `default_rng(1)` and rerun. The gap changes size, but which score is larger should not.